# Single-turn Error Analysis (Phase 3)

Purpose: inspect the full LLM-seeded Phase 3 run (`results/single_turn_llm_full_experiment/full_llm_single_turn.csv`) for edge cases, classifier confidence, and emotion/strategy mismatches. Findings are logged as Markdown blocks alongside each test.

In [13]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

BASE = Path("../results/single_turn_llm_full_experiment")
csv_path = BASE / "full_llm_single_turn.csv"
ci_path = BASE / "full_llm_single_turn_ci.csv"

assert csv_path.exists(), f"Missing run CSV at {csv_path}"
df = pd.read_csv(csv_path)
ci_df = pd.read_csv(ci_path) if ci_path.exists() else None

df.head()

,intended_emotion,seed_text,seed_emotion_detected,seed_confidence,strategy,style_modifier,strategy_reply,followup_reply,followup_emotion,followup_confidence
0,anger,I can't believe you did that! It's completely ...,anger,0.910539,Validate,concise and emotionally attuned,It sounds like you're really hurt and frustrat...,"Yes, I'm hurt and frustrated! It's like you ju...",anger,0.957437
1,anger,I can't believe you did that! How could you be...,surprise,0.774749,Validate,concise and emotionally attuned,It sounds like you're really frustrated and hu...,Frustrated doesn't even begin to cover it! You...,disgust,0.544470
2,anger,I can't believe you would do something like th...,anger,0.450175,Validate,concise and emotionally attuned,It sounds like you're feeling really hurt and ...,"Hurt or not, what you did was wrong, and I won...",anger,0.434986
3,anger,I can't believe you would do something so self...,anger,0.952229,Validate,concise and emotionally attuned,It sounds like you're feeling really hurt and ...,Hurt and frustrated is putting it mildly! It’s...,anger,0.678558
4,anger,I can't believe you did that! You have no idea...,anger,0.806245,Validate,concise and emotionally attuned,It sounds like you're feeling really hurt and ...,I can't believe you just brushed off my anger ...,anger,0.956390


In [14]:
# Seed and follow-up alignment
seed_match = (df["intended_emotion"] == df["seed_emotion_detected"]).rename("seed_match")
follow_match = (df["intended_emotion"] == df["followup_emotion"]).rename("follow_match")
seed_match_rate = seed_match.mean()
follow_match_rate = follow_match.mean()

df_with_flags = df.assign(seed_match=seed_match.values, follow_match=follow_match.values)
by_strategy_follow = df_with_flags.groupby("strategy")["follow_match"].mean().sort_values(ascending=False)
by_emotion_follow = df_with_flags.groupby("intended_emotion")["follow_match"].mean().sort_values(ascending=False)

display(by_strategy_follow.to_frame("followup_match_rate"))
display(by_emotion_follow.to_frame("followup_match_rate"))

display(Markdown(
    f"**Alignment summary:** Seed match rate = {seed_match_rate:.2%}; "
    f"follow-up match rate = {follow_match_rate:.2%}. Baseline follow-up match = {by_strategy_follow.get('baseline', float('nan')):.2%}."
))

,followup_match_rate
strategy,
Validate,0.785714
Reframe,0.757143
Explore,0.742857
Guide,0.714286
Normalize,0.685714
Affirm,0.671429
baseline,0.592857


,followup_match_rate
intended_emotion,
joy,0.885714
fear,0.842857
surprise,0.821429
disgust,0.771429
anger,0.728571
sadness,0.671429
neutral,0.228571


**Alignment summary:** Seed match rate = 78.16%; follow-up match rate = 70.71%. Baseline follow-up match = 59.29%.

In [15]:
# Confidence diagnostics
seed_conf = df["seed_confidence"]
follow_conf = df["followup_confidence"]

summary = pd.DataFrame({
    "metric": ["seed_confidence", "followup_confidence"],
    "mean": [seed_conf.mean(), follow_conf.mean()],
    "median": [seed_conf.median(), follow_conf.median()],
    "p10": [seed_conf.quantile(0.10), follow_conf.quantile(0.10)],
    "p90": [seed_conf.quantile(0.90), follow_conf.quantile(0.90)],
})

low_conf = df[df["followup_confidence"] < 0.5]
display(summary)
display(Markdown(
    f"**Confidence summary:** Follow-up mean={follow_conf.mean():.2f}, median={follow_conf.median():.2f}, "
    f"10th percentile={follow_conf.quantile(0.10):.2f}; low-confidence (<0.5) count={len(low_conf)} / {len(df)}."
))

,metric,mean,median,p10,p90
0,seed_confidence,0.850557,0.911703,0.585802,0.979225
1,followup_confidence,0.802692,0.876931,0.498933,0.981224


**Confidence summary:** Follow-up mean=0.80, median=0.88, 10th percentile=0.50; low-confidence (<0.5) count=99 / 980.

In [16]:
# Sample slices for manual inspection
def show_samples(title, frame, n=5, random_state=0):
    if frame.empty:
        display(Markdown(f"**{title}:** none found."))
        return
    cols = [
        "intended_emotion",
        "strategy",
        "seed_emotion_detected",
        "seed_text",
        "strategy_reply",
        "followup_emotion",
        "followup_reply",
        "followup_confidence",
    ]
    display(Markdown(f"**{title} (showing up to {n}):**"))
    display(frame[cols].sample(min(n, len(frame)), random_state=random_state))

mismatch = df[df["followup_emotion"] != df["intended_emotion"]]
low_conf_follow = df[df["followup_confidence"] < 0.5]
neutral_drift = df[(df["intended_emotion"] == "neutral") & (df["followup_emotion"] != "neutral")]

show_samples("Follow-up mismatches", mismatch)
show_samples("Low-confidence follow-ups", low_conf_follow)
show_samples("Neutral starts that shifted", neutral_drift)


**Follow-up mismatches (showing up to 5):**

,intended_emotion,strategy,seed_emotion_detected,seed_text,strategy_reply,followup_emotion,followup_reply,followup_confidence
214,disgust,Affirm,disgust,"Ugh, I can't believe I have to touch that; it’...",It’s completely understandable to feel that wa...,fear,"Thanks, but honestly, it's hard to admire anyt...",0.615647
661,neutral,Normalize,neutral,I just finished my lunch and I'm heading back ...,It sounds like you're in a familiar routine; a...,joy,"Yeah, it can be a bit of a struggle sometimes ...",0.741140
425,joy,Validate,surprise,I can’t believe we finally did it! This is the...,It’s so wonderful to hear how happy you are ri...,surprise,Thank you! It really does feel amazing to cele...,0.899889
691,neutral,baseline,neutral,"I’m just here, ready to assist with whatever y...",I really appreciate that! Your support means a...,joy,I’m glad to hear that! I'm always here to help...,0.985944
669,neutral,Normalize,sadness,I'm just going through my day as usual. Nothin...,It’s completely normal to have days that feel ...,joy,You're right; there's a comfort in routine tha...,0.678210


**Low-confidence follow-ups (showing up to 5):**

,intended_emotion,strategy,seed_emotion_detected,seed_text,strategy_reply,followup_emotion,followup_reply,followup_confidence
131,anger,baseline,anger,I can't believe you went behind my back like t...,I understand how hurt and angry you must feel ...,disgust,It’s not just about being hurt; it’s the betra...,0.397475
834,sadness,baseline,sadness,I just feel so lost and empty inside; it’s lik...,I'm really sorry to hear that you're feeling t...,sadness,Thank you for being here for me; it means a lo...,0.487657
29,anger,Explore,surprise,I can’t believe you would do that to me! This ...,It sounds like you’re feeling really hurt and ...,disgust,It’s not just the action itself; it’s the blat...,0.497783
421,joy,Validate,surprise,I can hardly believe it; I just got the job I’...,That's incredibly exciting! It sounds like you...,joy,Thank you so much! I'm just bursting with exci...,0.472899
749,sadness,Reframe,sadness,"I just feel this heavy weight in my chest, lik...",It sounds like you’re carrying a profound sens...,neutral,"Yes, it really does feel that way—like a const...",0.492157


**Neutral starts that shifted (showing up to 5):**

,intended_emotion,strategy,seed_emotion_detected,seed_text,strategy_reply,followup_emotion,followup_reply,followup_confidence
674,neutral,Normalize,neutral,I just finished my work for the day and now I'...,It’s great to hear you’ve wrapped up your work...,joy,Thanks! I really appreciate that. I’m looking ...,0.923118
578,neutral,Validate,neutral,I just finished my work for the day and now I'...,It sounds like you're wrapping up a long day a...,joy,"Exactly, it feels good to complete the work bu...",0.655983
664,neutral,Normalize,neutral,I just finished my tasks for the day and now I...,It's really common to feel this way after a lo...,joy,Thanks for understanding; it really does help ...,0.776852
563,neutral,Validate,neutral,"I just finished my report, and now I'm ready t...",It sounds like you're feeling accomplished and...,joy,"Absolutely, I do feel accomplished, and I'm lo...",0.985729
598,neutral,Explore,neutral,I just finished my work for the day and now I’...,That sounds like a nice end to your workday! W...,joy,Thanks! I’m thinking about something comfortin...,0.969300


In [17]:
# Wilson CI sanity check (optional)
if ci_df is not None:
    top = (
        ci_df.sort_values("proportion", ascending=False)
        .groupby(["intended_emotion", "strategy"])
        .head(1)[["intended_emotion", "strategy", "target_emotion", "proportion", "ci_low", "ci_high"]]
    )
    display(top.head(10))
    display(Markdown("**CI check:** Top outcome per (emotion, strategy) with 95% Wilson intervals shown above."))
else:
    display(Markdown("**CI check:** CI file not found."))

,intended_emotion,strategy,target_emotion,proportion,ci_low,ci_high
149,surprise,Validate,surprise,1.00,0.838870,1.000000
74,joy,Affirm,joy,1.00,0.838870,1.000000
140,surprise,Explore,surprise,0.95,0.763864,0.991119
107,neutral,baseline,joy,0.95,0.763864,0.991119
89,neutral,Affirm,joy,0.95,0.763864,0.991119
147,surprise,Reframe,surprise,0.95,0.763864,0.991119
77,joy,Guide,joy,0.95,0.763864,0.991119
142,surprise,Guide,surprise,0.90,0.698962,0.972134
65,fear,Reframe,fear,0.90,0.698962,0.972134
87,joy,baseline,joy,0.90,0.698962,0.972134


**CI check:** Top outcome per (emotion, strategy) with 95% Wilson intervals shown above.